In [ ]:
import pandas as pd
df=pd.read_csv("notebooks/data/house_prices.csv")
df.head()
df.isnull().sum()
df.columns
df.head(10)
df.sample(10)
df.dtypes
df.nunique()
df.isnull().sum().sort_values(ascending=False)
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10,6))
sns.histplot(df["Amount(in rupees)"])
plt.show()
def parse_amount(x):
    if not isinstance(x, str):
        return None
    x = x.strip().lower()
    try:
        if "lac" in x:
            return float(x.replace("lac", "").strip()) * 100000

        if "cr" in x:
            return float(x.replace("cr", "").strip()) * 10000000

        return float(x.replace(",", ""))
    except:
        return None
df["price_clean"] = df["Amount(in rupees)"].apply(parse_amount)

df = df.dropna(subset=["price_clean"])
df[["Amount(in rupees)", "price_clean"]].head(20)
import matplotlib.pyplot as plt
import seaborn as sns

plt.figure(figsize=(10,6))
sns.histplot(df["price_clean"], log_scale=True)
plt.title("Price Distribution")
plt.show()
import re

def parse_area(x):
    if pd.isna(x):
        return None

    x = str(x).lower()

    num = re.findall(r"\d+\.?\d*", x)

    if not num:
        return None

    area = float(num[0])

    if "sqm" in x or "m²" in x:
        area *= 10.764

    return area
df["carpet_area_sqft"] = df["Carpet Area"].apply(parse_area)

df[["Carpet Area", "carpet_area_sqft"]].head(20)
def parse_floor(x):
    if pd.isna(x):
        return None

    x = str(x).lower()

    if "ground" in x:
        return 0

    match = re.search(r"\d+", x)

    if match:
        return int(match.group())

    return None
df["floor_num"] = df["Floor"].apply(parse_floor)

df[["Floor", "floor_num"]].head(20)
numeric_cols = ["Bathroom", "Balcony", "Car Parking"]

for col in numeric_cols:
    df[col] = pd.to_numeric(df[col], errors="coerce")
    df[col] = df[col].fillna(df[col].median())
top_locations = df["location"].value_counts().head(50).index

df["location_grouped"] = df["location"].apply(
    lambda x: x if x in top_locations else "other"
)
price_per_sqft = df["price_clean"] / df["carpet_area_sqft"]

low = price_per_sqft.quantile(0.01)
high = price_per_sqft.quantile(0.99)

df = df[(price_per_sqft >= low) & (price_per_sqft <= high)]
from sklearn.model_selection import train_test_split

numeric_features = [
    "carpet_area_sqft",
    "floor_num",
    "Bathroom",
    "Balcony"
]

categorical_features = [
    "location_grouped",
    "Furnishing",
    "Transaction",
    "Ownership",
    "facing"
]

X = df[numeric_features + categorical_features]
y = df["price_clean"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42
)
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

preprocessor = ColumnTransformer([
    (
        "num",
        Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler())
        ]),
        numeric_features
    ),
    (
        "cat",
        Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore"))
        ]),
        categorical_features
    )
])
from sklearn.linear_model import LinearRegression

lr_model = Pipeline([
    ("prep", preprocessor),
    ("reg", LinearRegression())
])

lr_model.fit(X_train, y_train)
from sklearn.ensemble import RandomForestRegressor

rf_model = Pipeline([
    ("prep", preprocessor),
    ("reg", RandomForestRegressor(
        n_estimators=200,
        random_state=42
    ))
])

rf_model.fit(X_train, y_train)
from sklearn.metrics import (
    mean_absolute_error,
    root_mean_squared_error,
    r2_score
)

models = {
    "Linear Regression": lr_model,
    "Random Forest": rf_model
}

for name, model in models.items():
    pred = model.predict(X_test)

    print("=" * 40)
    print(name)
    print("MAE :", mean_absolute_error(y_test, pred))
    print("RMSE:", root_mean_squared_error(y_test, pred))
    print("R² :", r2_score(y_test, pred))
pred = rf_model.predict(X_test)

plt.figure(figsize=(6,6))
plt.scatter(y_test, pred, alpha=0.4)
plt.xlabel("Actual Price")
plt.ylabel("Predicted Price")
plt.title("Actual vs Predicted")
plt.show()
import joblib

joblib.dump(rf_model, "house_price.pkl")
import json

locations = sorted(df["location_grouped"].unique())

with open("locations.json", "w") as f:
    json.dump(locations, f)
print(df.columns)
locations = sorted(df["location"].dropna().unique().tolist())

import json
with open("../backend/locations.json", "w") as f:
    json.dump(locations, f)

print("Done")
print(model)